# Prepare Data - Prediksi Kualitas Udara Jakarta

ubah 5 file ISPU mentah di data/raw menjadi data/processed/data_siap.csv

In [1]:
import pandas as pd
from config import (DATA_RAW_DIR, DATA_RAW_POLA, DATA_SIAP, RINGKASAN_DATA,
                    FITUR, TARGET, KOLOM_ID, kategori_ispu)

ISPU_MAKS_WAJAR = 500


def buang(df, syarat_buang, alasan):
    jumlah = int(syarat_buang.sum())
    catatan_buang.append((alasan, jumlah))
    print(f"  - {alasan}: {jumlah} baris dibuang")
    return df[~syarat_buang]

## 1. Baca & gabungkan 5 file stasiun

In [2]:
daftar_file = sorted(DATA_RAW_DIR.glob(DATA_RAW_POLA))
if len(daftar_file) != 5:
    raise SystemExit(f"Harusnya ada 5 file di {DATA_RAW_DIR}, ketemu {len(daftar_file)}.")

df_mentah = pd.concat([pd.read_csv(f) for f in daftar_file], ignore_index=True)
jumlah_awal = len(df_mentah)
print("File:", [f.name for f in daftar_file])
print("Jumlah baris:", jumlah_awal)
df_mentah.head()

File: ['ispu_dki1.csv', 'ispu_dki2.csv', 'ispu_dki3.csv', 'ispu_dki4.csv', 'ispu_dki5.csv']
Jumlah baris: 25865


,tanggal,stasiun,pm25,pm10,so2,co,o3,no2,max,critical,categori
0,2010-01-01,DKI1 (Bunderan HI),NaN,60.0,4.0,73.0,27.0,14.0,73.0,CO,SEDANG
1,2010-01-02,DKI1 (Bunderan HI),NaN,32.0,2.0,16.0,33.0,9.0,33.0,O3,BAIK
2,2010-01-03,DKI1 (Bunderan HI),NaN,27.0,2.0,19.0,20.0,9.0,27.0,PM10,BAIK
3,2010-01-04,DKI1 (Bunderan HI),NaN,22.0,2.0,16.0,15.0,6.0,22.0,PM10,BAIK
4,2010-01-05,DKI1 (Bunderan HI),NaN,25.0,2.0,17.0,15.0,8.0,25.0,PM10,BAIK


## 2. Rapikan tipe data

In [3]:
df_rapi = df_mentah.copy()
df_rapi["tanggal"] = pd.to_datetime(df_rapi["tanggal"], errors="coerce")
df_rapi["max"] = pd.to_numeric(df_rapi["max"], errors="coerce")
df_rapi["stasiun"] = df_rapi["stasiun"].astype("string").str.strip()
df_rapi["categori"] = df_rapi["categori"].astype("string").str.strip().str.upper()
df_rapi.dtypes

tanggal     datetime64[us]
stasiun             string
pm25               float64
pm10               float64
so2                float64
co                 float64
o3                 float64
no2                float64
max                float64
critical               str
categori            string
dtype: object

### Mencari data bermasalah

In [4]:
print(df_rapi["categori"].value_counts(dropna=False))
print("max = 0    :", (df_rapi["max"] == 0).sum())
print("max kosong :", df_rapi["max"].isna().sum())

categori
SEDANG                16043
BAIK                   4213
TIDAK SEHAT            2992
TIDAK ADA DATA         2409
SANGAT TIDAK SEHAT      207
BERBAHAYA                 1
Name: count, dtype: int64[pyarrow]
max = 0    : 2350
max kosong : 59


## 3. Buang baris tidak valid

In [5]:
df = df_rapi.copy()
catatan_buang = []

jumlah_awal = len(df)
print("Buang baris tidak valid:")

df = buang(df, df["tanggal"].isna() | df["stasiun"].isna(),
           "tanggal/stasiun kosong")

df = buang(df, df.duplicated(subset=["tanggal", "stasiun"], keep="first"),
           "baris double (tanggal & stasiun sama)")

tidak_ada_data = (df["categori"] == "TIDAK ADA DATA") | df["max"].isna() | (df["max"] <= 0)
df = buang(df, tidak_ada_data,
           "tidak ada pengukuran (TIDAK ADA DATA / max 0 / kosong)")

df = buang(df, df["max"] > ISPU_MAKS_WAJAR,
           f"ISPU di atas {ISPU_MAKS_WAJAR} (tidak wajar)")

tidak_cocok = df["max"].map(kategori_ispu) != df["categori"]
df = buang(df, tidak_cocok,
           "angka max & label kategori tidak cocok (data rusak)")

jumlah_valid = len(df)
print("Sisa baris valid:", jumlah_valid)

Buang baris tidak valid:
  - tanggal/stasiun kosong: 0 baris dibuang
  - baris double (tanggal & stasiun sama): 0 baris dibuang
  - tidak ada pengukuran (TIDAK ADA DATA / max 0 / kosong): 2409 baris dibuang
  - ISPU di atas 500 (tidak wajar): 0 baris dibuang
  - angka max & label kategori tidak cocok (data rusak): 168 baris dibuang
Sisa baris valid: 23288


## 4. Bentuk fitur & target per stasiun

In [6]:
potongan = []
for nama_stasiun, data_stasiun in df.groupby("stasiun"):
    ispu = data_stasiun.set_index("tanggal").sort_index()["max"].asfreq("D")
    tabel = pd.DataFrame({"tanggal": ispu.index, "stasiun": nama_stasiun})
    tabel["ispu_hari_ini"] = ispu.values
    tabel["ispu_kemarin"] = ispu.shift(1).values
    tabel["bulan"] = ispu.index.month
    tabel["hari"] = ispu.index.day_of_week # 0 = Senin, 6 = Minggu
    tabel[TARGET] = ispu.shift(-1).values
    potongan.append(tabel)

semua = pd.concat(potongan, ignore_index=True)
data_siap = semua[KOLOM_ID + FITUR + [TARGET]]

### Cek lompatan 2021 > 2023: `ispu_kemarin` di 1 Januari 2023 harus kosong (NaN) di semua stasiun.

In [7]:
data_siap.head(6)

,tanggal,stasiun,ispu_hari_ini,ispu_kemarin,bulan,hari,ispu_besok
0,2010-01-01,DKI1 (Bunderan HI),73.0,NaN,1,4,33.0
1,2010-01-02,DKI1 (Bunderan HI),33.0,73.0,1,5,27.0
2,2010-01-03,DKI1 (Bunderan HI),27.0,33.0,1,6,22.0
3,2010-01-04,DKI1 (Bunderan HI),22.0,27.0,1,0,25.0
4,2010-01-05,DKI1 (Bunderan HI),25.0,22.0,1,1,30.0
5,2010-01-06,DKI1 (Bunderan HI),30.0,25.0,1,2,41.0


In [8]:
data_siap[data_siap["tanggal"] == "2023-01-01"]

,tanggal,stasiun,ispu_hari_ini,ispu_kemarin,bulan,hari,ispu_besok
4748,2023-01-01,DKI1 (Bunderan HI),55.0,NaN,1,6,52.0
9979,2023-01-01,DKI2 (Kelapa Gading),61.0,NaN,1,6,59.0
15207,2023-01-01,DKI3 (Jagakarsa),51.0,NaN,1,6,51.0
20438,2023-01-01,DKI4 (Lubang Buaya),57.0,NaN,1,6,53.0
24915,2023-01-01,DKI5 (Kebon Jeruk),52.0,NaN,1,6,48.0


## 5. Buang baris yang belum lengkap + rapikan

In [9]:
data_siap = data_siap.dropna(subset=FITUR + [TARGET])
print(f"{jumlah_valid - len(data_siap)} baris valid tidak dipakai karena "
      f"kemarin/besoknya bolong -> sisa {len(data_siap)} baris")

for kolom in ["ispu_hari_ini", "ispu_kemarin", TARGET]:
    if kolom in data_siap.columns:
        data_siap[kolom] = data_siap[kolom].astype(int)

data_siap = data_siap.sort_values(["tanggal", "stasiun"]).reset_index(drop=True)

453 baris valid tidak dipakai karena kemarin/besoknya bolong -> sisa 22835 baris


In [10]:
data_siap.head()

,tanggal,stasiun,ispu_hari_ini,ispu_kemarin,bulan,hari,ispu_besok
0,2010-01-02,DKI1 (Bunderan HI),33,73,1,5,27
1,2010-01-03,DKI1 (Bunderan HI),27,33,1,6,22
2,2010-01-04,DKI1 (Bunderan HI),22,27,1,0,25
3,2010-01-05,DKI1 (Bunderan HI),25,22,1,1,30
4,2010-01-06,DKI1 (Bunderan HI),30,25,1,2,41


## 6. Cek otomatis

In [11]:
assert list(data_siap.columns) == KOLOM_ID + FITUR + [TARGET], "Nama/urutan kolom tidak sesuai kontrak"
assert data_siap[FITUR + [TARGET]].notna().all().all(), "Masih ada sel kosong"
assert data_siap.duplicated(subset=KOLOM_ID).sum() == 0, "Ada tanggal+stasiun dobel"
for kolom in ["ispu_hari_ini", "ispu_kemarin", TARGET]:
    if kolom in data_siap.columns:
        assert data_siap[kolom].between(1, ISPU_MAKS_WAJAR).all(), f"{kolom} di luar 1-{ISPU_MAKS_WAJAR}"
if "bulan" in data_siap.columns:
    assert data_siap["bulan"].between(1, 12).all(), "bulan di luar 1-12"
if "hari" in data_siap.columns:
    assert data_siap["hari"].between(0, 6).all(), "hari di luar 0-6"
print("Semua pengecekan lolos")

Semua pengecekan lolos


## 7. Simpan + ringkasan

In [12]:
DATA_SIAP.parent.mkdir(parents=True, exist_ok=True)
data_siap.to_csv(DATA_SIAP, index=False)

kategori_besok = data_siap[TARGET].map(kategori_ispu).value_counts()
per_stasiun = data_siap.groupby("stasiun").agg(
    baris=("tanggal", "size"), mulai=("tanggal", "min"), akhir=("tanggal", "max"))

baris = [
    "RINGKASAN DATA - prepare_data.ipynb",
    "=" * 60,
    "Sumber : Kaggle 'Air Quality Index in Jakarta (2010-2021)' oleh senadu34",
    "         (asal data: Satu Data Jakarta), ispu_dki1.csv s.d. ispu_dki5.csv",
    f"Periode: {data_siap['tanggal'].min().date()} s.d. {data_siap['tanggal'].max().date()}",
    f"Stasiun: {data_siap['stasiun'].nunique()}",
    "Catatan: tahun 2022 tidak tersedia di dataset per stasiun",
    "",
    f"Baris awal (5 file)  : {jumlah_awal}",
    *[f"  dibuang - {alasan}: {n}" for alasan, n in catatan_buang],
    f"Baris valid          : {jumlah_valid}",
    f"Baris data_siap.csv  : {len(data_siap)}",
    "",
    f"Kolom : {list(data_siap.columns)}",
    "",
    "Per stasiun:",
    *[f"  {s:<22} {r.baris:>5} baris ({r.mulai.date()} s.d. {r.akhir.date()})"
      for s, r in per_stasiun.iterrows()],
    "",
    "Distribusi kategori ISPU besok (data tidak seimbang):",
    *[f"  {k:<20} {n:>6} ({n / len(data_siap):.1%})" for k, n in kategori_besok.items()],
]
teks = "\n".join(baris)
RINGKASAN_DATA.write_text(teks, encoding="utf-8")

print(f"Disimpan: {DATA_SIAP}")
print(teks)

Disimpan: C:\AI-AOL\aol-ai-prediksi-udara\data\processed\data_siap.csv
RINGKASAN DATA - prepare_data.ipynb
Sumber : Kaggle 'Air Quality Index in Jakarta (2010-2021)' oleh senadu34
         (asal data: Satu Data Jakarta), ispu_dki1.csv s.d. ispu_dki5.csv
Periode: 2010-01-02 s.d. 2025-02-27
Stasiun: 5
Catatan: tahun 2022 tidak tersedia di dataset per stasiun

Baris awal (5 file)  : 25865
  dibuang - tanggal/stasiun kosong: 0
  dibuang - baris double (tanggal & stasiun sama): 0
  dibuang - tidak ada pengukuran (TIDAK ADA DATA / max 0 / kosong): 2409
  dibuang - ISPU di atas 500 (tidak wajar): 0
  dibuang - angka max & label kategori tidak cocok (data rusak): 168
Baris valid          : 23288
Baris data_siap.csv  : 22835

Kolom : ['tanggal', 'stasiun', 'ispu_hari_ini', 'ispu_kemarin', 'bulan', 'hari', 'ispu_besok']

Per stasiun:
  DKI1 (Bunderan HI)      4898 baris (2010-01-02 s.d. 2025-02-27)
  DKI2 (Kelapa Gading)    4725 baris (2010-11-05 s.d. 2025-02-27)
  DKI3 (Jagakarsa)        4629 b